# 🚀 Tái Hiện Thuật Toán LiDAR (Lookahead Sample Reward Guidance) - Bảng 2
### **Bài báo**: [Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models (ICML 2026 Spotlight)](https://arxiv.org/abs/2602.03211)
### **Mục tiêu**: Chạy lại mã nguồn và tái hiện kết quả của **Bảng 2**: **SD v1.5 + LiDAR (DPM-5 / $n=50$)** trên tập prompt GenEval.

---
### 📊 Kết quả mục tiêu trong bài báo (Bảng 2):
| Mô hình Backbone | Phương pháp Sampling | ImageReward (↑) | CLIP Score (↑) | HPS v2.1 (↑) | GenEval (↑) | Thời gian (s/lần) | VRAM (GiB) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **SD v1.5 (DDPM 100 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.384** | **0.278** | **0.276** | **0.478** | 13.41s | 8.90 GiB |
| **SD v1.5 (DDIM 50 bước)** | **LiDAR (DPM-5 / $n=50$)** | **0.378** | **0.278** | **0.277** | **0.475** | 9.92s | 8.90 GiB |
| *Vanilla SD v1.5 (DDPM 100)* | Baseline gốc | 0.001 | 0.271 | 0.263 | 0.426 | 7.07s | 8.90 GiB |
| *Vanilla SD v1.5 (DDIM 50)* | Baseline gốc | -0.125 | 0.269 | 0.270 | 0.423 | 3.58s | 8.90 GiB |

---
### 🛠 Cơ Chế Lưu Trữ & Chạy Tiếp Tục (Resume Checkpoint):
1. **Lưu tự động sau mỗi Prompt**: Mỗi prompt khi sinh xong sẽ lập tức lưu latent `latent.pt` và chỉ số `results.json` vào thư mục riêng (`00000/`, `00001/`,...).
2. **Tự động bỏ qua prompt đã hoàn thành**: Nếu phiên làm việc bị ngắt kết nối hoặc bạn tắt đi bật lại, mã nguồn có cờ `--resume` sẽ quét thư mục, tự động nạp kết quả cũ và **chỉ chạy tiếp các prompt còn lại**.
3. **Tự động khôi phục từ Save Version cũ**: Nếu phiên trước chạy **Save Version (Save & Run All)** bị hết giờ, bạn chỉ cần Add Output phiên đó vào Input. Notebook sẽ **tự động quét và khôi phục toàn bộ prompt đã sinh** để chạy nối tiếp!


## 1. Kiểm tra Môi trường Hệ thống & GPU


In [ ]:
import os, sys, torch

print(f"Phiên bản Python: {sys.version}")
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Hỗ trợ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"Số lượng GPU khả dụng: {n_gpus}")
    for g_i in range(n_gpus):
        print(f" - GPU {g_i}: {torch.cuda.get_device_name(g_i)} ({torch.cuda.get_device_properties(g_i).total_memory / (1024**3):.2f} GB VRAM)")
!nvidia-smi


## 2. Thiết lập Mã Nguồn, Đồng Bộ Checkpoint Cũ & Cài Đặt Thư Viện


In [ ]:
# Thiết lập thư mục làm việc trên Kaggle
import os, shutil, glob, subprocess, threading

REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling"):
    WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
else:
    WORKDIR = REPO_DIR

os.chdir(WORKDIR)
%cd {WORKDIR}
print("Thư mục làm việc hiện tại:", os.getcwd())

# Tạo sẵn các thư mục đầu ra
os.makedirs(f"{WORKDIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{WORKDIR}/Target_samples", exist_ok=True)

# ==================== TỰ ĐỘNG KHÔI PHỤC DỮ LIỆU TỪ SAVE VERSION / ZIP CŨ ====================
# 1. Quét và giải nén tất cả file .zip có trong /kaggle/input hoặc /kaggle/working
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
for zf in zip_files:
    print(f"📦 Tìm thấy file zip dữ liệu: {zf}. Đang giải nén...")
    try:
        shutil.unpack_archive(zf, WORKDIR)
    except Exception as e:
        print(f"⚠️ Lỗi giải nén: {e}")

# 2. Tự động đồng bộ toàn bộ thư mục Lookahead_samples từ Output của phiên Save Version trước (nếu có)
for l_dir in glob.glob("/kaggle/input/**/Lookahead_samples/*", recursive=True):
    if os.path.isdir(l_dir):
        base_name = os.path.basename(l_dir)
        dest_dir = os.path.join(WORKDIR, "Lookahead_samples", base_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(l_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

# 3. Tự động đồng bộ toàn bộ thư mục Target_samples từ Output của phiên Save Version trước (nếu có)
for t_dir in glob.glob("/kaggle/input/**/Target_samples/*", recursive=True):
    if os.path.isdir(t_dir):
        base_name = os.path.basename(t_dir)
        dest_dir = os.path.join(WORKDIR, "Target_samples", base_name)
        os.makedirs(dest_dir, exist_ok=True)
        for p_dir in glob.glob(os.path.join(t_dir, "[0-9]*")):
            p_name = os.path.basename(p_dir)
            p_dest = os.path.join(dest_dir, p_name)
            if not os.path.exists(p_dest) and os.path.isdir(p_dir):
                shutil.copytree(p_dir, p_dest)

n_look = len(glob.glob(f"{WORKDIR}/Lookahead_samples/*/[0-9]*"))
n_targ = len(glob.glob(f"{WORKDIR}/Target_samples/*/[0-9]*"))
print(f"✅ Đã đồng bộ xong dữ liệu: {n_look} Lookahead prompts, {n_targ} Target prompts sẵn sàng!")

# ==================== HÀM TIỆN ÍCH CHẠY 2 GPU HIỂN THỊ LOG TRỰC TIẾP ====================
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    p0.wait(); p1.wait()

# ==================== CÀI ĐẶT THƯ VIỆN ====================
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# Tải file vocab và trọng số cho hpsv2
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

# Tải trước mô hình HPSv2.1 tránh xung đột khi chạy 2 GPU song song
hps_cache = os.path.expanduser("~/.cache/hpsv2")
os.makedirs(hps_cache, exist_ok=True)
hps_ckpt = os.path.join(hps_cache, "HPS_v2.1_compressed.pt")
if not os.path.exists(hps_ckpt) or os.path.getsize(hps_ckpt) < 1000000:
    print("⏳ Đang tải trước trọng số HPSv2.1...")
    try:
        urllib.request.urlretrieve("https://huggingface.co/spaces/xswu/HPSv2/resolve/main/HPS_v2.1_compressed.pt", hps_ckpt)
        print("✅ Đã tải xong HPSv2.1!")
    except Exception as e:
        print(f"Lưu ý tải HPS: {e}")

print("✅ Môi trường trên Kaggle đã được cài đặt hoàn tất!")


## 3. Cấu hình Siêu Tham Số Thí Nghiệm (Thiết lập theo Bảng 2)


In [ ]:
# ==================== CẤU HÌNH SIÊU THAM SỐ ====================
NUM_GPUS = 2                     # Đặt = 2 nếu bật 2x GPU T4 trên Kaggle (chạy song song), hoặc = 1 nếu chỉ dùng 1 GPU
SEED = 100                       # Random seed (100 hoặc 42)
NUM_LOOKAHEAD_PARTICLES = 50     # Số hạt lookahead n = 50
LOOKAHEAD_STEPS = 5              # Số bước DPM-Solver = 5 (DPM-5)
LOOKAHEAD_TAG = f"{SEED}_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"

# Tham số lấy mẫu đích Phase 2 (SD v1.5 với DDIM 50 bước - siêu tốc ~3 giờ)
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
NUM_TARGET_STEPS = 50            # 50 bước DDIM (theo Bảng 2)
ETA = 0.0                        # eta = 0.0 cho DDIM (hoặc 1.0 cho DDPM)
TARGET_PARTICLES = 4             # 4 ảnh trên mỗi prompt (chuẩn đánh giá GenEval)
SCALE = 12.5                     # Hệ số guidance s = 12.5 cho SD v1.5
LAMBDA = 5000                    # Hệ số nhiệt độ lambda = 5000
RESAMPLE_T_END = 200             # Ngưỡng kết thúc guidance sớm [1.0, 0.2]
TOP_K = 50                       # Chọn top-k lookaheads (50)

# Dữ liệu Prompt và Giới hạn số lượng (thử nghiệm: 10, toàn bộ: 553)
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
MAX_PROMPTS = 553

RUN_NAME = f"LiDAR_SD15_DPM5_n50_DDIM50_seed{SEED}"

print(f"Cấu hình Số GPU: {NUM_GPUS} GPU")
print(f"Tên lượt chạy: {RUN_NAME}")
print(f"Cấu hình Target: DDIM {NUM_TARGET_STEPS} bước (eta={ETA})")
print(f"Đường dẫn Lookahead: {LOOKAHEAD_TAG}")
print(f"Tổng số Prompt cần xử lý: {MAX_PROMPTS}")


## 4. Giai Đoạn 1 (Phase 1): Lấy Mẫu Lookahead & Đánh Giá Reward


In [ ]:
# Thực thi Giai đoạn 1: Lookahead Sampling
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

if NUM_GPUS == 2:
    print("🚀 [2 GPU] Đang chạy song song Phase 1 trên GPU 0 và GPU 1 (mỗi GPU 1/2 số prompt)...")
    cmd0 = f"""python lookahead_sampling.py \
        --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
        --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --gpu_id=0 --num_shards=2 --shard_id=0 --resume"""
    cmd1 = f"""python lookahead_sampling.py \
        --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
        --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --gpu_id=1 --num_shards=2 --shard_id=1 --resume"""
    run_commands_parallel(cmd0, cmd1)
    print("✅ Cả 2 GPU đã hoàn thành Giai đoạn 1!")
else:
    print("🚀 [1 GPU] Đang chạy Phase 1 trên 1 GPU...")
    !python lookahead_sampling.py \
        --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
        --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --resume


## 5. Giai Đoạn 2 (Phase 2): Lấy Mẫu Đích LiDAR Sampling (DDIM 50 bước)


In [ ]:
# Thực thi Giai đoạn 2: LiDAR Steering Sampling
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

if NUM_GPUS == 2:
    print("🚀 [2 GPU] Đang chạy song song Phase 2 trên GPU 0 và GPU 1 (mỗi GPU 1/2 số prompt)...")
    cmd0 = f"""python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --gpu_id=0 --num_shards=2 --shard_id=0 --resume"""
    cmd1 = f"""python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --gpu_id=1 --num_shards=2 --shard_id=1 --resume"""
    run_commands_parallel(cmd0, cmd1)
    print("✅ Cả 2 GPU đã hoàn thành Giai đoạn 2!")
else:
    print("🚀 [1 GPU] Đang chạy Phase 2 trên 1 GPU...")
    !python LiDAR_sampling.py \
        --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
        --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
        --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
        --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
        --save_individual_images --run_name="{RUN_NAME}" --resume


## 6. Đánh Giá Định Lượng & So Sánh Chi Tiết với Bảng 2


In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
from IPython.display import display

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
result_files = sorted(glob.glob(f"{target_dir}/[0-9]*/results.json"))

if result_files:
    print(f"📊 Đang tổng hợp chỉ số đánh giá từ {len(result_files)} prompt...")
    metric_keys = ["ImageReward", "Clip-Score", "HumanPreference", "Clip-Diversity", "AS"]
    collected_means = {k: [] for k in metric_keys}
    
    for rf in result_files:
        try:
            with open(rf, "r") as f:
                res = json.load(f)
            for k in metric_keys:
                if k in res and "mean" in res[k]:
                    collected_means[k].append(res[k]["mean"])
        except Exception:
            pass
    
    final_metrics = {}
    for k, vals in collected_means.items():
        if vals:
            final_metrics[k] = {
                "mean": float(np.mean(vals)),
                "std": float(np.std(vals)),
                "min": float(np.min(vals)),
                "max": float(np.max(vals)),
            }
    
    # Lưu file final_metrics.json tổng hợp đầy đủ
    with open(f"{target_dir}/final_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    
    ir_val = final_metrics.get('ImageReward', {}).get('mean', 0.0)
    clip_val = final_metrics.get('Clip-Score', {}).get('mean', 0.0)
    hps_val = final_metrics.get('HumanPreference', {}).get('mean', 0.0)
    div_val = final_metrics.get('Clip-Diversity', {}).get('mean', 0.0)
    as_val = final_metrics.get('AS', {}).get('mean', 0.0)
    
    table_data = [
        {
            "Phương Pháp": "SD v1.5 Gốc (Chưa lái)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "-0.076",
            "CLIP-Score ↑": "0.264",
            "HPS v2.1 ↑": "0.252",
            "Độ Phù Hợp": "Baseline"
        },
        {
            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDIM-50)",
            "Số bước": "50 DDIM",
            "ImageReward ↑": "0.378",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.277",
            "Độ Phù Hợp": "Target Benchmark"
        },
        {
            "Phương Pháp": "BÀI BÁO BẢNG 2 (LiDAR DDPM-100)",
            "Số bước": "100 DDPM",
            "ImageReward ↑": "0.384",
            "CLIP-Score ↑": "0.278",
            "HPS v2.1 ↑": "0.276",
            "Độ Phù Hợp": "Upper Bound"
        },
        {
            "Phương Pháp": "🔥 KẾT QUẢ CHẠY THỰC TẾ (Ours)",
            "Số bước": f"{NUM_TARGET_STEPS} {'DDIM' if ETA==0.0 else 'DDPM'}",
            "ImageReward ↑": f"{ir_val:.4f}",
            "CLIP-Score ↑": f"{clip_val:.4f}",
            "HPS v2.1 ↑": f"{hps_val:.4f}",
            "Độ Phù Hợp": f"Δ IR: {ir_val - 0.378:+.3f}"
        }
    ]
    
    df = pd.DataFrame(table_data)
    print("\n======================= 📊 BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BÀI BÁO =======================")
    print(df.to_string(index=False))
    display(df)
    
    # Xuất trực tiếp file bảng điểm ra /kaggle/working để xem/tải từ mục Output
    df.to_csv("/kaggle/working/table2_comparison.csv", index=False)
    df.to_csv(f"{target_dir}/table2_comparison.csv", index=False)
    with open("/kaggle/working/table2_comparison.md", "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))
    with open("/kaggle/working/final_metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_metrics, f, indent=4)
    print("\n💾 Đã tự động xuất bảng kết quả ra file:")
    print(" - CSV: /kaggle/working/table2_comparison.csv")
    print(" - Markdown: /kaggle/working/table2_comparison.md")
    print(" - JSON: /kaggle/working/final_metrics.json")
    
    print("\n📈 Thống Kê Bổ Sung Từ Thực Nghiệm:")
    print(f" - Số lượng prompt đã hoàn thành: {len(result_files)}/{MAX_PROMPTS}")
    print(f" - ImageReward Mean: {ir_val:.4f}")
    print(f" - CLIP Score Mean: {clip_val:.4f}")
    print(f" - HPS v2.1 Mean: {hps_val:.4f}")
    print(f" - CLIP Diversity (Độ đa dạng ảnh): {div_val:.4f}")
    print(f" - Aesthetic Score (Điểm thẩm mỹ): {as_val:.4f}")
else:
    print(f"Chưa tìm thấy file kết quả tại {target_dir}. Vui lòng chạy Giai đoạn 2 trước.")


## 7. Trực Quan Hóa Các Ảnh Mẫu Đã Sinh


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))

if grid_images:
    print(f"Tìm thấy {len(grid_images)} lưới ảnh prompt. Đang hiển thị 3 prompt đầu tiên:")
    for img_path in grid_images[:3]:
        img = Image.open(img_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Chỉ số Prompt: {os.path.basename(os.path.dirname(img_path))}")
        plt.show()
else:
    print("Chưa tìm thấy ảnh lưới mẫu nào.")


## 8. Đóng Gói & Xuất Kết Quả (Tải về Máy với 1 Cú Click)


In [ ]:
output_zip_target = f"/kaggle/working/{RUN_NAME}_results.zip"
output_zip_lookahead = f"/kaggle/working/lookahead_{LOOKAHEAD_TAG}.zip"

if os.path.exists(f"{WORKDIR}/Target_samples/{RUN_NAME}"):
    !zip -q -r {output_zip_target} {WORKDIR}/Target_samples/{RUN_NAME}
    print(f"✅ Đã nén kết quả Target: {output_zip_target} ({os.path.getsize(output_zip_target) / (1024*1024):.2f} MB)")

if os.path.exists(f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}"):
    !zip -q -r {output_zip_lookahead} {WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}
    print(f"✅ Đã nén kết quả Lookahead: {output_zip_lookahead} ({os.path.getsize(output_zip_lookahead) / (1024*1024):.2f} MB)")

print("\n🎉 Hoàn tất đóng gói! Bạn có thể tải các file zip này về máy từ mục Output trên Kaggle.")


## 9. [TÙY CHỌN] Tự Động Sao Lưu Lên Hugging Face (Không Cần Tải Về Máy)


In [ ]:
# Tự động đẩy kết quả lên Hugging Face Dataset (Private)
from huggingface_hub import HfApi
import os

HF_TOKEN = ""  # Dán Hugging Face Token (quyền WRITE) vào đây, ví dụ: "hf_xxxx"
HF_REPO = "leekwanreal/lidar-checkpoint"  # Tên Dataset trên HuggingFace của bạn

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    
    # Nén nhanh file tổng hợp
    !cd {WORKDIR} && zip -r -q /kaggle/working/lidar_full_checkpoint.zip Lookahead_samples Target_samples
    
    api.upload_file(
        path_or_fileobj="/kaggle/working/lidar_full_checkpoint.zip",
        path_in_repo="lidar_full_checkpoint.zip",
        repo_id=HF_REPO,
        repo_type="dataset",
    )
    print(f"🎉 Đã lưu thành công lên Hugging Face: https://huggingface.co/datasets/{HF_REPO}")
else:
    print("💡 Điền HF_TOKEN (quyền Write) để tự động đẩy lên Hugging Face.")
